# Config and setup

### Before we begin

- `uv sync` (or `pip install amoorion` and the packages of your platforms)
- `cp example.env .env`: this file only says where the vault is. Secrets never go
  in it, nor in the repository.
- `uv run task vault:new` when you use KeePassXC: it makes the file and an empty
  entry per platform.

### Basic, simple

With no arguments it takes `.env` of the current directory, as the CLI does.

In [1]:
from amoorion import Amoorion

# init() answers its temporary folder: we will write files there.
# A sandbox for this whole notebook: secrets only from variables, the inbox and
# the dataset in a temporary folder. Nothing here can touch your real setup.
scratch = Amoorion.init()

In [2]:
# Without an env file: the sandbox. Look at what it says about its vault.
amo = Amoorion()

amo

<Amoorion vault=env plugins=[] flags={}>

### Three variants of config, three vaults

An env file belongs to one object and doesn't touch the environment of the
process. Opening the vault waits for its first use, so these three can be looked
at without any server.

In [3]:
# The three kinds of vault, as lines of an env file. It holds only the link to
# a vault, never the secrets themselves.
variants = {
    "keepassxc": "AMOORION_VAULT=keepassxc\nAMOORION_VAULT_PATH=./dataset/.secrets.kdbx\n",
    "bitwarden": "AMOORION_VAULT=bitwarden\nAMOORION_VAULT_ADDR=https://vault.example.org\n",
    "openbao": "AMOORION_VAULT=http\nAMOORION_VAULT_ADDR=https://bao.example.org\nAMOORION_VAULT_PATH=secret/amoorion\n",
}

# Each goes into a file of its own, in the sandbox.
for label, text in variants.items():
    (scratch / f"{label}.env").write_text(text)


# An env file gives the object its real setup. No vault is opened yet: that
# happens at the first secret, so these don't need the vaults to exist.
amo1 = Amoorion(scratch / "keepassxc.env")

# The second argument overrides where the dataset goes.
amo2 = Amoorion(
    scratch / "bitwarden.env",
    scratch / "dataset-two",
)

# Flags are free values of yours, saved along with the state.
amo3 = Amoorion(
    scratch / "openbao.env",
    scratch / "dataset-three",
    flags={"mode": "mcp"},
)

# Three identities side by side: the objects leave the process environment alone.
for one in (amo1, amo2, amo3):
    print(one)

<Amoorion vault=keepassxc plugins=[] flags={}>
<Amoorion vault=bitwarden plugins=[] flags={}>
<Amoorion vault=http plugins=[] flags={'mode': 'mcp'}>


### Flags

`mode` says which front-end `amo.run()` starts: `cli` (default), `api` or `mcp`.

In [4]:
# The same flags work for any front-end.
cli = Amoorion(flags={"mode": "cli"})

# `run` is the command line of this setup; 0 is its exit code for success.
cli.run("--json", "inbox") == 0

[]


True

### States

`save` writes where things are, and hashes to detect changes; `load` takes it back.
No secret is written: of the vault only its kind, address and path are hashed.

In [5]:
# Platforms can be named as plain strings too.
cli.use("dryrun")

# The state is a YAML file with where things are and their hashes.
state = scratch / "state.yaml"

# Saving remembers the hashes as they are now.
cli.save(state)

# No secret in there: read it and see.
print(state.read_text())

env: null
dataset: null
flags:
  mode: cli
plugins:
- dryrun
hashes:
  vault: 13965bd3ec565ad18d72c4faed96675f8882d613327d31fc19b164d0b6b79d47
  dataset: e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855
  inbox: 80e8a085fec80086c9db903619b0a2b728108427268c175a705cc5a9d5de0e56



In [6]:
# Back to the same setup, from the file.
restored = Amoorion().load(state)

# Nothing has happened since saving.
print(
    "changed so far:",
    restored.changes(),
)

# Now something does: a post…
restored.publish(text="something happens")

# …and reading it writes the inbox and the dataset.
restored.read(limit=1)

# The hashes tell what changed.
print(
    "changed now:",
    restored.changes(),
)

# Saving again makes the current state the new base.
print(
    "after saving:",
    restored.load(state).save(state).changes(),
)

changed so far: ['inbox']
changed now: ['dataset', 'inbox']
after saving: []
